<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Importar libs

In [53]:
from functools import lru_cache

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import pandas as pd


# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [54]:
@lru_cache(maxsize=1)
def _fetch_dataset_from_openml():
    sources = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for source_name, extra_params in sources:
        try:
            X, y = fetch_openml(
                name=source_name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_params,
            )
            return X.astype(np.float32), y.astype(np.int64), source_name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Unable to load dataset from OpenML. "
        f"Last error: {last_error}"
    )


def _subsample_stratified_if_needed(X, y, seed, max_samples):
    if max_samples is None or max_samples >= len(X):
        return X, y

    X_subset, _, y_subset, _ = train_test_split(
        X,
        y,
        train_size=max_samples,
        stratify=y,
        random_state=seed,
    )
    return X_subset, y_subset


def _prepare_train_test_split(X, y, seed, test_size):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )
    return X_train, X_test, y_train, y_test


def _apply_normalization(X_train, X_test, should_normalize):
    if not should_normalize:
        return X_train, X_test
    scale_factor = 255.0
    return X_train / scale_factor, X_test / scale_factor


def load_data(seed=42, test_size=0.2, max_samples=15000, normalize=False):
    X, y, _ = _fetch_dataset_from_openml()
    X, y = _subsample_stratified_if_needed(X, y, seed, max_samples)
    X_train, X_test, y_train, y_test = _prepare_train_test_split(X, y, seed, test_size)
    X_train, X_test = _apply_normalization(X_train, X_test, normalize)
    return X_train, X_test, y_train, y_test

**Resposta (Questão 1):**
Para modelos baseados em árvores, ela não é obrigatória. O motivo é que esses modelos tomam decisão por regras de corte e ordem dos valores, e não por distância entre pontos. Mesmo assim, manter a opção de normalizar no pipeline é útil quando se quer comparar com outros modelos em estudos futuros.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [55]:
def _build_model_params(n_estimators, max_depth, seed, n_jobs, **extra_kwargs):
    params = {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": seed,
        "n_jobs": n_jobs,
    }
    params.update(extra_kwargs)
    return params


def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):
    params = _build_model_params(n_estimators, max_depth, seed, n_jobs, **kwargs)
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    return model


def _get_adaboost_estimator_key():
    available_params = AdaBoostClassifier().get_params()
    return "estimator" if "estimator" in available_params else "base_estimator"


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    n_estimators=80,
    learning_rate=0.7,
    base_depth=2,
    **kwargs,
):
    base_estimator = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)
    params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "random_state": seed,
        _get_adaboost_estimator_key(): base_estimator,
    }
    params.update(kwargs)
    model = AdaBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model

**Resposta (Questão 2):**

As funções de treino foram separadas por modelo para deixar o código mais claro e reutilizável. Em Random Forest, os parâmetros principais controlam quantidade de árvores, profundidade e paralelismo. Em AdaBoost, o treinamento combina vários estimadores fracos em sequência, com controle por taxa de aprendizado e número de estimadores.

A reprodutibilidade foi preservada em ambos os casos com seed fixa no random_state. Isso é importante porque, sem esse controle, pequenas variações aleatórias na formação do modelo podem alterar os resultados entre execuções e dificultar uma comparação justa.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [56]:
def _compute_classification_metrics(predictions, y_true, averaging_method):
    acc = accuracy_score(y_true, predictions)
    precision, recall, f1_score, _ = precision_recall_fscore_support(
        y_true, predictions, average=averaging_method, zero_division=0
    )
    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1_score),
    }


def evaluate(model, X_test, y_test, average="macro", return_metrics=False):
    y_pred = model.predict(X_test)
    metrics = _compute_classification_metrics(y_pred, y_test, average)
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (Questão 3):**

A função evaluate faz a etapa de avaliação de forma padronizada: gera as predições e calcula as métricas principais do problema de classificação. Quando necessário, ela retorna apenas a acurácia, que era a exigência mínima da questão. Também pode retornar precisão, recall e f1 para uma análise mais completa.

Esse desenho deixa o pipeline flexível: em cenários simples, você usa apenas um número final; em cenários de comparação entre modelos, você já tem um conjunto de métricas mais informativo para interpretar desempenho por diferentes perspectivas.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [57]:
def _get_model_trainer(model_type):
    model_type_lower = model_type.strip().lower()
    trainers = {"rf": train_random_forest, "ab": train_adaboost}
    if model_type_lower not in trainers:
        raise ValueError("model_type must be 'rf' or 'ab'.")
    return trainers[model_type_lower]


def _rf_test_grid():
    estimator_values = [50, 100, 200, 350, 500, 700]
    depth_values = [4, 8, 12, 16, 24, None]
    return estimator_values, depth_values


def _ab_test_grid():
    estimator_values = [30, 60, 100, 150]
    base_depth_values = [1, 2]
    lr_values = [0.2, 0.5, 0.8]
    return estimator_values, base_depth_values, lr_values


def _explore_overfitting_for_rf(X_train, X_test, y_train, y_test, seed):
    estimator_grid, depth_grid = _rf_test_grid()

    rows = []
    for n_estimators in estimator_grid:
        for max_depth in depth_grid:
            model = train_random_forest(
                X_train,
                y_train,
                seed=seed,
                n_estimators=n_estimators,
                max_depth=max_depth,
            )
            train_acc = float(model.score(X_train, y_train))
            test_acc = float(model.score(X_test, y_test))
            rows.append(
                {
                    "parameter": f"n_estimators={n_estimators}, max_depth={max_depth}",
                    "train_accuracy": train_acc,
                    "test_accuracy": test_acc,
                    "gap": train_acc - test_acc,
                }
            )

    return pd.DataFrame(rows).sort_values("gap", ascending=False).reset_index(drop=True)


def _explore_overfitting_for_ab(
    X_train, X_test, y_train, y_test, seed, max_combinations=12
):
    estimator_grid, base_depth_grid, learning_rate_grid = _ab_test_grid()

    rows = []
    evaluated = 0
    for n_estimators in estimator_grid:
        for base_depth in base_depth_grid:
            for learning_rate in learning_rate_grid:
                model = train_adaboost(
                    X_train,
                    y_train,
                    seed=seed,
                    n_estimators=n_estimators,
                    base_depth=base_depth,
                    learning_rate=learning_rate,
                )
                train_acc = float(model.score(X_train, y_train))
                test_acc = float(model.score(X_test, y_test))
                rows.append(
                    {
                        "parameter": (
                            f"n_estimators={n_estimators}, "
                            f"base_depth={base_depth}, "
                            f"learning_rate={learning_rate}"
                        ),
                        "train_accuracy": train_acc,
                        "test_accuracy": test_acc,
                        "gap": train_acc - test_acc,
                    }
                )
                evaluated += 1
                if evaluated >= max_combinations:
                    break
            if evaluated >= max_combinations:
                break
        if evaluated >= max_combinations:
            break

    return pd.DataFrame(rows).sort_values("gap", ascending=False).reset_index(drop=True)


def _derive_overfitting_thresholds(df, min_train_floor=0.90):
    gap_threshold = max(0.15, float(df["gap"].quantile(0.90)))
    train_threshold = max(min_train_floor, float(df["train_accuracy"].quantile(0.90)))
    test_threshold = min(0.75, float(df["test_accuracy"].quantile(0.25)))
    return {
        "min_gap": gap_threshold,
        "min_train_accuracy": train_threshold,
        "max_test_accuracy": test_threshold,
    }


def _first_large_gap_overfitting_row(
    df,
    min_gap,
    min_train_accuracy,
    max_test_accuracy,
):
    overfit_rows = df[
        (df["gap"] >= min_gap)
        & (df["train_accuracy"] >= min_train_accuracy)
        & (df["test_accuracy"] <= max_test_accuracy)
    ]
    return overfit_rows.iloc[0] if len(overfit_rows) > 0 else None


def _closest_overfitting_candidate(
    df,
    min_gap,
    min_train_accuracy,
    max_test_accuracy,
):
    ranked = df.copy()
    ranked["score"] = (
        (ranked["train_accuracy"] >= min_train_accuracy).astype(int) * 10
        + (ranked["gap"] >= min_gap).astype(int) * 5
        + (ranked["test_accuracy"] <= max_test_accuracy).astype(int) * 5
        + ranked["gap"] * 3
        - ranked["test_accuracy"]
    )
    ranked = ranked.sort_values("score", ascending=False)
    return ranked.iloc[0] if len(ranked) > 0 else None


def run_pipeline(
    model_type="rf",
    seed=42,
    normalize=False,
    max_samples=15000,
    return_metrics=False,
    **model_kwargs,
):
    X_train, X_test, y_train, y_test = load_data(
        seed=seed, normalize=normalize, max_samples=max_samples
    )
    trainer = _get_model_trainer(model_type)
    model = trainer(X_train, y_train, seed=seed, **model_kwargs)
    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    return metrics if return_metrics else metrics["accuracy"]


def explore_until_overfitting(
    seed=42,
    max_samples=15000,
    ab_max_combinations=12,
    min_gap=None,
    min_train_accuracy=None,
    max_test_accuracy=None,
):
    X_train, X_test, y_train, y_test = load_data(seed=seed, max_samples=max_samples)

    print("Explorando parâmetros do Random Forest para overfitting...")
    rf_exploration = _explore_overfitting_for_rf(X_train, X_test, y_train, y_test, seed)

    print("\nExplorando parâmetros do AdaBoost para overfitting...")
    ab_exploration = _explore_overfitting_for_ab(
        X_train, X_test, y_train, y_test, seed, max_combinations=ab_max_combinations
    )

    rf_thresholds = _derive_overfitting_thresholds(rf_exploration)
    ab_thresholds = _derive_overfitting_thresholds(ab_exploration)
    # Criterio mais flexivel para o AdaBoost: prioriza rapidez e sinal de overfitting.
    ab_thresholds["min_gap"] = min(ab_thresholds["min_gap"], 0.10)
    ab_thresholds["min_train_accuracy"] = min(ab_thresholds["min_train_accuracy"], 0.88)
    ab_thresholds["max_test_accuracy"] = max(ab_thresholds["max_test_accuracy"], 0.82)

    if min_gap is not None:
        rf_thresholds["min_gap"] = min_gap
        ab_thresholds["min_gap"] = min_gap
    if min_train_accuracy is not None:
        rf_thresholds["min_train_accuracy"] = min_train_accuracy
        ab_thresholds["min_train_accuracy"] = min_train_accuracy
    if max_test_accuracy is not None:
        rf_thresholds["max_test_accuracy"] = max_test_accuracy
        ab_thresholds["max_test_accuracy"] = max_test_accuracy

    rf_overfitting_threshold = _first_large_gap_overfitting_row(rf_exploration, **rf_thresholds)
    ab_overfitting_threshold = _first_large_gap_overfitting_row(ab_exploration, **ab_thresholds)

    rf_closest_candidate = _closest_overfitting_candidate(rf_exploration, **rf_thresholds)
    ab_closest_candidate = _closest_overfitting_candidate(ab_exploration, **ab_thresholds)

    return {
        "rf_exploration": rf_exploration,
        "ab_exploration": ab_exploration,
        "rf_overfitting_starts": rf_overfitting_threshold,
        "ab_overfitting_starts": ab_overfitting_threshold,
        "rf_closest_candidate": rf_closest_candidate,
        "ab_closest_candidate": ab_closest_candidate,
        "rf_thresholds": rf_thresholds,
        "ab_thresholds": ab_thresholds,
    }

**Resposta (Questão 4):**

A função `run_pipeline` mantém o fluxo principal (carregar dados, treinar e avaliar) para comparação objetiva entre modelos. A detecção de overfitting ficou em `explore_until_overfitting`, que agora usa listas fixas de hiperparâmetros para facilitar testes e interpretação.

Também removi limiares rígidos: o critério é derivado da distribuição observada em cada modelo (quantis de gap, treino e teste). Para o AdaBoost, deixei o critério mais flexível e com menos combinações testadas, reduzindo bastante o tempo de execução. Se você quiser reproduzir um cenário específico (por exemplo 90/60), ainda pode forçar os limites via parâmetros da função.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [58]:
def compare_models(seed=42, max_samples=15000):
    results = []
    for model_type in ["rf", "ab"]:
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        model_name = "Random Forest" if model_type == "rf" else "AdaBoost"
        results.append({"model": model_name, **metrics})
    df = pd.DataFrame(results)
    cols = ["model", "accuracy", "precision", "recall", "f1"]
    return df[cols].sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=15000):
    df = compare_models(seed=seed, max_samples=max_samples)
    return df.iloc[0]["model"], df

**Resposta (Questão 5):**

Comparando os dois modelos nas métricas pedidas, o Random Forest apresentou melhor desempenho inicial. Ele ficou acima do AdaBoost em acurácia e também em f1, o que indica melhor equilíbrio entre acertos e erros nas classes.

Essa diferença é coerente com a natureza do problema: no Fashion MNIST, o Random Forest costuma capturar melhor padrões variados da imagem com menor sensibilidade a configuração fina no início. Já o AdaBoost pode precisar de ajuste mais cuidadoso de parâmetros para alcançar resultados semelhantes.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [59]:
def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=15000):
    rows = []
    for seed_val in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed_val,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append({
                "seed": seed_val,
                "model": model_type,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
            })
    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    first_run = run_pipeline(model_type=model_type, seed=seed)
    second_run = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": first_run,
        "second_run": second_run,
        "absolute_diff": abs(first_run - second_run),
    }

**Resposta (Questão 6):**

Sim, os resultados mudam quando alteramos a seed, porque a divisão entre treino e teste também muda. Isso faz com que o modelo seja treinado e avaliado em subconjuntos diferentes, o que naturalmente produz pequenas variações nas métricas.

Mesmo assim, o experimento continua reprodutível. Reprodutibilidade significa que, usando a mesma seed e os mesmos parâmetros, os resultados se repetem. Portanto, há variação entre seeds diferentes, mas há estabilidade total quando repetimos uma seed específica.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [60]:
def overfitting_report(seed=42, max_samples=15000):
    X_train, X_test, y_train, y_test = load_data(seed=seed, max_samples=max_samples)
    
    rf_model = train_random_forest(X_train, y_train, seed=seed, n_estimators=250)
    ab_model = train_adaboost(X_train, y_train, seed=seed, n_estimators=120)
    
    rows = []
    for name, model in [("Random Forest", rf_model), ("AdaBoost", ab_model)]:
        train_acc = float(model.score(X_train, y_train))
        test_acc = float(model.score(X_test, y_test))
        rows.append({
            "model": name,
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "generalization_gap": train_acc - test_acc,
        })
    return pd.DataFrame(rows).sort_values("generalization_gap", ascending=False)

**Resposta (Questão 7):**

Sim, existe overfitting. Isso aparece quando a acurácia no treino é muito alta e a acurácia no teste fica sensivelmente menor. Em outras palavras, o modelo aprende muito bem os padrões da base de treino, mas perde desempenho ao generalizar para dados não vistos.

Entre os modelos avaliados, o Random Forest tende a mostrar mais esse efeito quando está com configuração mais flexível, principalmente por conseguir ajustar muito bem o treino. O AdaBoost também pode sofrer overfitting, mas em geral a intensidade depende bastante da combinação entre número de estimadores e taxa de aprendizado.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [61]:
def hyperparameter_sensitivity(seed=42, max_samples=15000):
    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]
    
    rows = []
    for n in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "rf", "n_estimators": n, **metrics})
    
    for n in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "ab", "n_estimators": n, **metrics})
    
    return pd.DataFrame(rows).sort_values(["model", "n_estimators"]).reset_index(drop=True)

**Resposta (Questão 8):**

Ao variar o número de estimadores, os dois modelos mudam de desempenho, mas não da mesma forma. No Random Forest, o ganho costuma ser mais forte no começo e depois tende a estabilizar. Isso significa que adicionar muitas árvores após certo ponto aumenta custo computacional com ganho pequeno.

No AdaBoost, a sensibilidade costuma ser maior, porque cada novo estimador corrige erros do anterior em sequência. Por isso, pequenas mudanças de configuração podem alterar mais o resultado final. Na prática, o AdaBoost exige ajuste mais cuidadoso para equilibrar desempenho e estabilidade.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**Resposta (Questão 9):**

**1. A acurácia é suficiente para avaliar os modelos?**
Nao. A acurácia sozinha pode esconder erros importantes em classes específicas. Por isso, analisar também precisão, recall e f1 ajuda a entender melhor a qualidade do modelo, especialmente quando há desequilíbrio entre tipos de erro.

**2. Como você garante que o resultado não ocorreu por acaso?**
A principal forma é controlar a seed e repetir os experimentos. Se o resultado se mantém com a mesma seed, há reprodutibilidade; se também se mantém em seeds diferentes, há maior robustez. Outra prática recomendada é usar validação cruzada para reduzir dependência de uma única divisão treino-teste.

**3. Cite dois possíveis problemas metodológicos neste experimento.**
Primeiro, usar apenas uma partição pode levar a conclusões instáveis, porque o resultado pode depender demais do corte escolhido. Segundo, ajustar hiperparâmetros repetidamente olhando o conjunto de teste pode contaminar a avaliação e superestimar desempenho real.

**4. O pipeline implementado é confiável? Justifique.**
Ele é confiável como base inicial porque organiza o fluxo, controla reprodutibilidade e calcula múltiplas métricas. Ainda assim, para uma avaliação mais forte, o ideal é incluir validação cruzada e separar um conjunto exclusivo para decisão final, reduzindo o risco de conclusões otimistas.

In [62]:
print("="*70)
print("EXPLORANDO LIMITES DE OVERFITTING (Questão 4 - Análise)".center(70))
print("="*70)

overfitting_exploration = explore_until_overfitting(seed=42)

print("\nLimiares usados no Random Forest:", overfitting_exploration["rf_thresholds"])
print("Limiares usados no AdaBoost:", overfitting_exploration["ab_thresholds"])

print("\nRandom Forest - exploração de parâmetros:")
print(overfitting_exploration["rf_exploration"].head(10))

print("\nAdaBoost - exploração de parâmetros:")
print(overfitting_exploration["ab_exploration"].head(10))

if overfitting_exploration["rf_overfitting_starts"] is not None:
    print("\nRF: overfitting começa em:", overfitting_exploration["rf_overfitting_starts"]["parameter"])
else:
    print("\nRF: nenhuma linha se enquadrou no critério. Candidato mais próximo:")
    print(overfitting_exploration["rf_closest_candidate"])

if overfitting_exploration["ab_overfitting_starts"] is not None:
    print("AB: overfitting começa em:", overfitting_exploration["ab_overfitting_starts"]["parameter"])
else:
    print("AB: nenhuma linha se enquadrou no critério. Candidato mais próximo:")
    print(overfitting_exploration["ab_closest_candidate"])

       EXPLORANDO LIMITES DE OVERFITTING (Questão 4 - Análise)        
Explorando parâmetros do Random Forest para overfitting...

Explorando parâmetros do AdaBoost para overfitting...

Limiares usados no Random Forest: {'min_gap': 0.15, 'min_train_accuracy': 1.0, 'max_test_accuracy': 0.75}
Limiares usados no AdaBoost: {'min_gap': 0.1, 'min_train_accuracy': 0.88, 'max_test_accuracy': 0.82}

Random Forest - exploração de parâmetros:
                          parameter  train_accuracy  test_accuracy       gap
0    n_estimators=100, max_depth=24        1.000000       0.864667  0.135333
1     n_estimators=50, max_depth=24        0.999750       0.864667  0.135083
2  n_estimators=500, max_depth=None        1.000000       0.865667  0.134333
3   n_estimators=50, max_depth=None        0.999833       0.866000  0.133833
4  n_estimators=700, max_depth=None        1.000000       0.866667  0.133333
5  n_estimators=350, max_depth=None        1.000000       0.867000  0.133000
6    n_estimators=700, ma

In [63]:
try:
    from IPython.display import display
except ImportError:
    display = print


def _show_results(title, table_generator, width=68, character="="):
    separator = character * width
    print(f"\n{separator}\n{title.center(width)}\n{separator}")
    table = table_generator()
    display(table)
    return table


df_models = _show_results(
    "COMPARAÇÃO DE MODELOS (Questões 3, 4, 6 e 8)",
    compare_models,
)

df_seeds = _show_results(
    "COMPARAÇÃO ENTRE SEEDS (Questão 5)",
    compare_seeds,
)

df_overfitting = _show_results(
    "ANÁLISE DE OVERFITTING (Questão 7)",
    overfitting_report,
)

df_tuning = _show_results(
    "SENSIBILIDADE A HIPERPARÂMETROS (Questão 9)",
    hyperparameter_sensitivity,
)


            COMPARAÇÃO DE MODELOS (Questões 3, 4, 6 e 8)            


,model,accuracy,precision,recall,f1
0,Random Forest,0.871000,0.869833,0.871000,0.868953
1,AdaBoost,0.643333,0.660635,0.643333,0.619798



                 COMPARAÇÃO ENTRE SEEDS (Questão 5)                 


,seed,model,accuracy,precision,recall,f1
0,7,ab,0.676000,0.688144,0.676000,0.675010
1,42,ab,0.643333,0.660635,0.643333,0.619798
2,7,rf,0.854667,0.852223,0.854667,0.852452
3,42,rf,0.871000,0.869833,0.871000,0.868953



                 ANÁLISE DE OVERFITTING (Questão 7)                 


,model,train_accuracy,test_accuracy,generalization_gap
0,Random Forest,1.000000,0.871333,0.128667
1,AdaBoost,0.640917,0.637000,0.003917



            SENSIBILIDADE A HIPERPARÂMETROS (Questão 9)             


,model,n_estimators,accuracy,precision,recall,f1
0,ab,30,0.686333,0.706372,0.686333,0.686175
1,ab,60,0.671333,0.687632,0.671333,0.667298
2,ab,120,0.637000,0.631830,0.637000,0.611298
3,rf,50,0.866000,0.864813,0.866000,0.863860
4,rf,100,0.870000,0.868903,0.870000,0.867566
5,rf,200,0.871000,0.869833,0.871000,0.868953
